# 9.4 GBDT案例实战 - 产品定价模型

In [2]:
import pandas as pd
df = pd.read_excel(r"D:\1Ftostudy\商业数据分析\@Python大数据分析与机器学习商业案例实战\@Python大数据分析与机器学习商业案例实战\第9章 AdaBoost与GBDT模型\源代码汇总_Jupyer Notebook\产品定价模型.xlsx")
df.head()

,页数,类别,彩印,纸张,价格
0,207,技术类,0,双胶纸,60
1,210,技术类,0,双胶纸,62
2,206,技术类,0,双胶纸,62
3,218,技术类,0,双胶纸,64
4,209,技术类,0,双胶纸,60


In [6]:
df['类别'].value_counts()  # 分类统计 类别3类

类别
技术类    336
教辅类    333
办公类    331
Name: count, dtype: int64

In [8]:
df['彩印'].value_counts()    # 分类统计 彩印2类

彩印
0    648
1    352
Name: count, dtype: int64

In [10]:
df['纸张'].value_counts()     # 分类统计 纸张3类

纸张
双胶纸    615
铜版纸    196
书写纸    189
Name: count, dtype: int64

In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['类别'] = le.fit_transform(df['类别'])  # 处理类别

In [14]:
# 将类别一列处理后，我们可以使用value_counts()方法查看转化效果：
df['类别'].value_counts()

类别
1    336
2    333
0    331
Name: count, dtype: int64

In [16]:
# 另外一种文本内容转为数值的方法，注意不要再运行完上面的代码后运行，因为上面的内容已经被替代完毕了，如果想尝试，需要重新运行，并且，先运行下面的代码
# df['类别'] = df['类别'].replace({'办公类': 0, '技术类': 1, '教辅类': 2})  
# df['类别'].value_counts()

In [18]:
# 下面我们使用同样的方法处理“纸张”一列：
le = LabelEncoder()
df['纸张'] = le.fit_transform(df['纸张'])

In [20]:
# 将纸张一列处理后，我们可以使用value_counts()方法查看转化效果：
df['纸张'].value_counts()

纸张
1    615
2    196
0    189
Name: count, dtype: int64

In [22]:
# 此时的表格如下：
df.head()

,页数,类别,彩印,纸张,价格
0,207,1,0,1,60
1,210,1,0,1,62
2,206,1,0,1,62
3,218,1,0,1,64
4,209,1,0,1,60


In [24]:
X = df.drop(columns='价格')   # 特征变量
y = df['价格']     # 目标变量

In [26]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

## 线性回归模型

In [30]:
#模型搭建
from sklearn.linear_model import LinearRegression

# 创建线性回归模型
linear_model = LinearRegression()

# 训练模型
linear_model.fit(X_train, y_train)

LinearRegression()

LinearRegression()

In [32]:
#模型预测及评估
# 模型预测
y_pred_linear = linear_model.predict(X_test)
print(y_pred_linear[0:50])

# 创建对比表格
a_linear = pd.DataFrame()
a_linear['预测值'] = list(y_pred_linear)
a_linear['实际值'] = list(y_test)
print(a_linear.head())

# 查看预测评分 - 方法1
linear_score = linear_model.score(X_test, y_test)
print(f"线性回归模型评分: {linear_score}")

# 查看预测评分 - 方法2
from sklearn.metrics import r2_score
r2_linear = r2_score(y_test, y_pred_linear)
print(f"线性回归R2分数: {r2_linear}")

# 查看系数重要性（线性回归没有feature_importances_，用coef_代替）
features = X.columns
coefficients = linear_model.coef_

# 通过DataFrame展示特征系数
coef_df = pd.DataFrame()
coef_df['特征名称'] = features
coef_df['系数值'] = coefficients
coef_df['绝对值系数'] = abs(coefficients)
coef_df = coef_df.sort_values('绝对值系数', ascending=False)
print(coef_df)

[59.54252245 64.6381339  55.69413372 90.09807885 68.20439221 42.46231683
 54.60170224 72.48233971 39.87013978 80.37661576 62.38649807 61.45935391
 63.97588807 70.62805138 61.59180307 56.85333807 61.08497536 53.54210891
 72.25764242 70.76050055 48.25401144 76.29366055 62.25404891 85.85296948
 91.10499824 76.64407096 57.1814064  62.70407035 57.43894183 50.99586976
 48.50230455 49.67137809 96.12819737 60.2970164  76.43920947 60.88998306
 49.91330748 67.0049868  61.2940666  99.63441883 56.05190703 60.36692244
 38.0388204  58.57517724 43.86658372 43.46923622 63.6781516  81.74704532
 53.93945641 62.8834566 ]
         预测值  实际值
0  59.542522   75
1  64.638134   84
2  55.694134   68
3  90.098079   90
4  68.204392   85
线性回归模型评分: 0.47857387412755525
线性回归R2分数: 0.47857387412755525
  特征名称       系数值     绝对值系数
1   类别 -9.380922  9.380922
2   彩印  7.337378  7.337378
3   纸张  3.248686  3.248686
0   页数  0.132449  0.132449


## 随机森林回归模型

In [35]:
#模型搭建
from sklearn.ensemble import RandomForestRegressor

# 创建随机森林回归模型
rf_model = RandomForestRegressor(random_state=123, n_estimators=100)

# 训练模型
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=123)

RandomForestRegressor(random_state=123)

## 模型预测及评估

In [38]:
# 模型预测
y_pred_rf = rf_model.predict(X_test)
print(y_pred_rf[0:50])

# 创建对比表格
a_rf = pd.DataFrame()
a_rf['预测值'] = list(y_pred_rf)
a_rf['实际值'] = list(y_test)
print(a_rf.head())

# 查看预测评分 - 方法1
rf_score = rf_model.score(X_test, y_test)
print(f"随机森林模型评分: {rf_score}")

# 查看预测评分 - 方法2
r2_rf = r2_score(y_test, y_pred_rf)
print(f"随机森林R2分数: {r2_rf}")

# 查看特征重要性
rf_importances = rf_model.feature_importances_

# 通过DataFrame的方式展示特征重要性
importances_rf_df = pd.DataFrame()
importances_rf_df['特征名称'] = features
importances_rf_df['特征重要性'] = rf_importances
importances_rf_df = importances_rf_df.sort_values('特征重要性', ascending=False)
print(importances_rf_df)

[ 72.          83.23190476  68.96        89.98        84.36633333
  37.04        37.04333333  63.805       54.12        74.74
  79.87966667  78.39366667  82.59090476  59.2122381   79.532
  40.79        54.          30.45116667  95.05566667  59.72890476
  33.305      102.          79.79966667  81.753       91.78
  47.78        49.6075      77.211       50.15        64.33
  38.99733333  61.31809524  96.7         49.08        68.02
  52.82296825  34.73833333  85.737       52.74       103.58
  48.13833333  49.19        34.61        51.61        35.31
  35.31        51.76        73.32921429  31.56616667  52.54      ]
         预测值  实际值
0  72.000000   75
1  83.231905   84
2  68.960000   68
3  89.980000   90
4  84.366333   85
随机森林模型评分: 0.8889051002734795
随机森林R2分数: 0.8889051002734795
  特征名称     特征重要性
0   页数  0.507501
1   类别  0.415148
2   彩印  0.043245
3   纸张  0.034105


## 支持向量机回归模型

In [41]:
#模型搭建
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# 数据标准化（SVM对数据尺度敏感）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 创建支持向量机回归模型
svr_model = SVR(kernel='rbf', C=1.0)

# 训练模型
svr_model.fit(X_train_scaled, y_train)

SVR()

SVR()

In [43]:
#模型预测及评估
# 模型预测
y_pred_svr = svr_model.predict(X_test_scaled)
print(y_pred_svr[0:50])

# 创建对比表格
a_svr = pd.DataFrame()
a_svr['预测值'] = list(y_pred_svr)
a_svr['实际值'] = list(y_test)
print(a_svr.head())

# 查看预测评分 - 方法1
svr_score = svr_model.score(X_test_scaled, y_test)
print(f"支持向量机模型评分: {svr_score}")

# 查看预测评分 - 方法2
r2_svr = r2_score(y_test, y_pred_svr)
print(f"支持向量机R2分数: {r2_svr}")

# 注意：SVM没有直接的特征重要性，但可以通过其他方法评估
print("SVM模型训练完成，特征重要性评估需要其他方法")

[70.32154218 81.64243274 66.60413023 88.65681796 75.86823301 48.35447331
 46.30832317 64.13253688 47.24872678 76.36460465 78.61118708 77.27245959
 80.78635131 61.31973779 77.46654923 47.21212334 53.71946578 46.01645413
 81.34814686 61.51604861 45.40912462 90.24890787 78.42287132 66.89437759
 82.69764965 58.85874097 55.37332448 75.90004627 50.00216828 60.69203658
 41.71936831 58.68200419 85.25401382 49.37784108 68.95713679 57.59329149
 40.57926846 77.87789718 53.32142762 91.20813342 48.79107681 52.83844864
 34.82596454 48.17442441 38.0900864  37.81684357 54.79844438 66.63123632
 46.11620531 54.26857524]
         预测值  实际值
0  70.321542   75
1  81.642433   84
2  66.604130   68
3  88.656818   90
4  75.868233   85
支持向量机模型评分: 0.8097228965429223
支持向量机R2分数: 0.8097228965429223
SVM模型训练完成，特征重要性评估需要其他方法
